# Full Stack Applications

Full-stack AI applications connect backend LLM and agent systems to real user interfaces. This notebook demonstrates the core patterns used to build production AI applications.

```
┌────────────────────────────────────────────────────────────────────────────┐
│                   Three-Layer Architecture                                 │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│  ┌───────────┐   REST/SSE/WS   ┌───────────┐   Python calls                │
│  │ Frontend  │ ──────────────► │  Backend  │ ─────────────►  ┌──────────┐  │
│  │ (React)   │ ◄────────────── │ (FastAPI) │ ◄─────────────  │  Agent   │  │
│  └───────────┘    Streaming    └───────────┘   Structured    └──────────┘  │
│                                                                            │
│  UI state         Session state / auth         Conversation mem            │
│  (ephemeral)      (persistent)                 (RAG context)               │
└────────────────────────────────────────────────────────────────────────────┘
```

**What this notebook covers:**
- Service layer pattern wrapping AI agents
- Streaming responses with Server-Sent Events (SSE)
- WebSocket bidirectional communication
- Background job processing with status polling
- Optimistic state updates
- Error handling and retry logic with tenacity
- Progress updates for long-running operations
- Frontend UX patterns (JavaScript reference)
- Database schema design for AI applications

**Setup:** Run `uv sync` to install dependencies, then set `OPENAI_API_KEY` in your `.env` file.


In [ ]:
import os
import json
import asyncio
import uuid
import hashlib
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import AsyncIterator, Optional

import nest_asyncio
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from sqlalchemy import (
    Column, String, Integer, Float, DateTime,
    ForeignKey, Text, JSON, create_engine
)
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type
)

nest_asyncio.apply()
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete!")


## Service Layer Pattern

The service layer wraps your agent logic in a clean interface. The backend shouldn't care whether you're using LangChain, raw OpenAI, or a local model — it just calls `agent_service.chat()` and receives a structured response. When you upgrade the underlying model or retrieval strategy, no backend routes or frontend code needs to change.

This isolation means you can swap the underlying model, change the retrieval strategy, or update prompts — without touching any backend routes or frontend code.


In [ ]:
SUPPORT_SYSTEM_PROMPT = """You are a helpful customer support agent for Acme Corp,
a fictional e-commerce company. Answer questions about orders, returns, shipping,
and products clearly and concisely. If you don't know something, say so honestly."""


@dataclass
class ChatResponse:
    content: str
    sources: list[str] = field(default_factory=list)
    confidence: float = 1.0


class AgentService:
    """Service layer wrapping agent operations with a clean interface."""

    def __init__(self, llm):
        self.llm = llm
        self.system_prompt = SUPPORT_SYSTEM_PROMPT

    async def chat(
        self, message: str, conversation_id: str = ""
    ) -> ChatResponse:
        """Return complete response (not streamed)."""
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=message),
        ]
        result = await self.llm.ainvoke(messages)
        return ChatResponse(content=result.content)

    async def chat_stream(
        self, message: str, conversation_id: str = ""
    ) -> AsyncIterator[str]:
        """Yield response chunks as they are generated."""
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=message),
        ]
        async for chunk in self.llm.astream(messages):
            if chunk.content:
                yield chunk.content


agent_service = AgentService(llm)
print("AgentService ready")
print(f"System prompt preview: {SUPPORT_SYSTEM_PROMPT[:60]}...")


In [ ]:
# Full (non-streaming) response
print("=" * 60)
print("FULL RESPONSE DEMO")
print("=" * 60)

response = asyncio.get_event_loop().run_until_complete(
    agent_service.chat("What is your return policy?")
)
print("Q: What is your return policy?")
print(f"A: {response.content}")
print(f"   sources={response.sources}, confidence={response.confidence}")

print()
print("=" * 60)
print("STREAMING RESPONSE DEMO")
print("=" * 60)


async def run_streaming_demo():
    print("Q: How do I track my order?")
    print("A: ", end="", flush=True)
    async for chunk in agent_service.chat_stream("How do I track my order?"):
        print(chunk, end="", flush=True)
    print()


asyncio.get_event_loop().run_until_complete(run_streaming_demo())


## Streaming with Server-Sent Events (SSE)

Streaming responses feel dramatically faster to users. A 10-second response appearing word-by-word feels faster than a 3-second response that appears all at once. Server-Sent Events (SSE) is the standard HTTP mechanism: a single long-lived connection where the server pushes `data:` lines to the client.

**Important detail:** JSON-encode each chunk on the server and parse it on the client to preserve newlines — raw newlines in SSE `data:` lines break the protocol.

In [ ]:
async def generate_sse_stream(message: str):
    """Simulate the SSE byte stream as it would appear over HTTP."""
    async for chunk in agent_service.chat_stream(message):
        if chunk:
            yield f"data: {json.dumps(chunk)}"
    yield "data: [DONE]"


async def run_sse_demo():
    message = "What payment methods do you accept?"
    print(f"SSE stream for: '{message}'\n")
    print("Raw SSE lines:")
    print("-" * 50)

    full_response = ""
    async for sse_line in generate_sse_stream(message):
        print(sse_line)  # show raw SSE format
        if sse_line != "data: [DONE]":
            chunk = json.loads(sse_line[6:])  # strip "data: "
            full_response += chunk

    print("-" * 50)
    print(f"\nReconstructed response:\n{full_response}")


asyncio.get_event_loop().run_until_complete(run_sse_demo())


## WebSocket Integration

Server-Sent Events work well for unidirectional streaming (server → client). WebSockets enable true bidirectional communication — use them when you need the client to send messages while the server is still streaming, or to push unsolicited notifications to clients without polling.

In [ ]:
class ConnectionManager:
    """Manages active WebSocket connections by user ID."""

    def __init__(self):
        self.active_connections: dict[str, object] = {}
        self._message_log: list[dict] = []

    async def connect(self, websocket, user_id: str):
        # In real FastAPI: await websocket.accept()
        self.active_connections[user_id] = websocket
        print(f"[CONNECT]    user={user_id} | active={list(self.active_connections)}")

    def disconnect(self, user_id: str):
        if user_id in self.active_connections:
            del self.active_connections[user_id]
            print(f"[DISCONNECT] user={user_id} | active={list(self.active_connections)}")

    async def send_message(self, user_id: str, message: dict):
        if user_id in self.active_connections:
            self._message_log.append({"to": user_id, **message})
            print(f"[SEND]       user={user_id} | {message}")


async def demo_websocket_lifecycle():
    manager = ConnectionManager()

    class MockWebSocket:
        pass

    print("=" * 55)
    print("WEBSOCKET LIFECYCLE DEMO")
    print("=" * 55)

    await manager.connect(MockWebSocket(), "user-001")
    await manager.connect(MockWebSocket(), "user-002")

    await manager.send_message("user-001", {"type": "chunk", "content": "Your order has "})
    await manager.send_message("user-001", {"type": "chunk", "content": "shipped!"})
    await manager.send_message("user-001", {"type": "done"})

    await manager.send_message("user-002", {"type": "notification", "content": "Return processed."})

    manager.disconnect("user-001")

    print(f"\nTotal messages delivered: {len(manager._message_log)}")


asyncio.get_event_loop().run_until_complete(demo_websocket_lifecycle())


## Background Job Processing

Some operations take too long for a synchronous request-response cycle: indexing uploaded documents, generating reports, running batch evaluations. The pattern is to accept the request, return a `job_id` immediately, and let clients poll for completion.

For production, use separate queues: a **fast queue** for chat, a **slow queue** for document indexing, and a **batch queue** for nightly processing. Workers are allocated to queues based on expected load patterns.

In [ ]:
@dataclass
class JobStatus:
    id: str
    status: str = "pending"   # pending, processing, completed, failed
    result: dict | None = None
    error: str | None = None
    created_at: datetime = field(default_factory=datetime.utcnow)


# In-memory job store (use Redis in production)
jobs: dict[str, JobStatus] = {}


async def simulate_document_indexing(job_id: str, filename: str, num_chunks: int = 42):
    """Simulate indexing a support knowledge base article."""
    await asyncio.sleep(0.1)
    jobs[job_id].status = "processing"
    print(f"  [{job_id[:8]}] processing — parsing '{filename}'")

    await asyncio.sleep(0.2)
    print(f"  [{job_id[:8]}] processing — embedding {num_chunks} chunks")

    await asyncio.sleep(0.1)
    jobs[job_id].status = "completed"
    jobs[job_id].result = {"chunks_indexed": num_chunks, "filename": filename}
    print(f"  [{job_id[:8]}] completed  — {num_chunks} chunks indexed")


def submit_job(job_fn, *args, **kwargs) -> str:
    """Submit a background job and return its ID immediately."""
    job_id = str(uuid.uuid4())
    jobs[job_id] = JobStatus(id=job_id, status="pending")
    asyncio.get_event_loop().create_task(job_fn(job_id, *args, **kwargs))
    return job_id


async def run_background_job_demo():
    print("=" * 55)
    print("BACKGROUND JOB DEMO")
    print("=" * 55)

    job_id = submit_job(
        simulate_document_indexing,
        "acme-return-policy.pdf",
        num_chunks=87,
    )
    print(f"Job submitted: {job_id[:8]}...")
    print(f"Initial status: {jobs[job_id].status}")
    print()

    # Poll until complete
    while jobs[job_id].status not in ("completed", "failed"):
        await asyncio.sleep(0.15)
        print(f"  Polling... status={jobs[job_id].status}")

    final = jobs[job_id]
    print(f"\nFinal status : {final.status}")
    print(f"Result       : {final.result}")


asyncio.get_event_loop().run_until_complete(run_background_job_demo())


## State Management

Full-stack AI applications manage state at multiple layers. **Client state** (which panel is open, draft text) is ephemeral — losing it on refresh is acceptable. **Server state** (conversations, progress, user data) must be persisted.

Optimistic updates bridge the gap: update the UI immediately as if the action succeeded, then roll back only if the server reports failure. This makes the app feel instant even when the backend is slow.

In [ ]:
@dataclass
class OptimisticMessage:
    id: str
    content: str
    status: str          # sending, sent, failed
    real_id: str | None = None
    error: str | None = None


class OptimisticMessageStore:
    """Implements optimistic UI updates: show immediately, rollback on failure."""

    def __init__(self):
        self.messages: list[OptimisticMessage] = []

    def add_optimistic(self, content: str) -> str:
        """Add message optimistically. Returns temp_id."""
        temp_id = f"temp-{uuid.uuid4().hex[:8]}"
        self.messages.append(
            OptimisticMessage(id=temp_id, content=content, status="sending")
        )
        return temp_id

    def confirm(self, temp_id: str, real_id: str) -> None:
        """Server confirmed the message — update to 'sent'."""
        for msg in self.messages:
            if msg.id == temp_id:
                msg.status = "sent"
                msg.real_id = real_id
                break

    def rollback(self, temp_id: str, error: str) -> None:
        """Server rejected — mark as failed so user can retry."""
        for msg in self.messages:
            if msg.id == temp_id:
                msg.status = "failed"
                msg.error = error
                break

    def display(self):
        icons = {"sending": "⏳", "sent": "✓", "failed": "✗"}
        for msg in self.messages:
            icon = icons.get(msg.status, "?")
            print(f"  [{icon} {msg.status:<7}] {msg.id} | '{msg.content[:45]}'")
            if msg.error:
                print(f"             Error: {msg.error}")


store = OptimisticMessageStore()

print("SUCCESS PATH: add_optimistic → confirm")
print("-" * 50)
temp_id = store.add_optimistic("I need to return my order #12345")
print("After add_optimistic:")
store.display()

store.confirm(temp_id, real_id="msg-server-789")
print("\nAfter confirm:")
store.display()

print()
print("FAILURE PATH: add_optimistic → rollback")
print("-" * 50)
temp_id2 = store.add_optimistic("What's my order status?")
print("After add_optimistic:")
store.display()

store.rollback(temp_id2, error="Network timeout — please retry")
print("\nAfter rollback:")
store.display()


## Error Handling & Retry Logic

AI applications face more diverse failure modes than traditional software: rate limits, context length exceeded, model timeouts, and genuine capability gaps. Good error handling distinguishes transient failures (retry automatically) from permanent ones (inform the user).

Three strategies work together:
1. **Exponential backoff** — wait longer after each consecutive failure
2. **Maximum retry count** — give up after N attempts
3. **Circuit breaking** — stop trying temporarily if too many requests fail in a window

In [ ]:
class RetryableError(Exception):
    """Transient error — safe to retry."""
    pass


class UserActionRequired(Exception):
    """Permanent error — user must intervene."""
    pass


# Track call count to simulate transient failures
_retry_call_count = 0


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=0.1, min=0.1, max=0.5),  # fast for demo
    retry=retry_if_exception_type(RetryableError),
)
async def call_agent_with_retry(message: str) -> str:
    global _retry_call_count
    _retry_call_count += 1
    print(f"  Attempt {_retry_call_count}...")

    if _retry_call_count < 3:
        raise RetryableError(f"Simulated transient failure (attempt {_retry_call_count})")

    # Third attempt succeeds — make real LLM call
    result = await llm.ainvoke([HumanMessage(content=message)])
    return result.content


async def run_retry_demo():
    global _retry_call_count
    _retry_call_count = 0

    print("=" * 55)
    print("RETRY DEMO (first 2 attempts fail, 3rd succeeds)")
    print("=" * 55)

    message = "How long does standard shipping take?"
    print(f"Question: '{message}'\n")

    try:
        result = await call_agent_with_retry(message)
        print(f"\nSucceeded on attempt {_retry_call_count}:")
        print(result)
    except RetryableError as e:
        print(f"\nAll retries exhausted: {e}")


asyncio.get_event_loop().run_until_complete(run_retry_demo())


## Progress Updates for Long-Running Operations

When generating a large document or processing a complex request takes 30+ seconds, users need feedback. A progress update stream — an async generator that yields stage milestones — keeps users engaged and informed rather than staring at a frozen spinner.

In [ ]:
@dataclass
class ProgressUpdate:
    stage: str
    message: str
    progress: float  # 0.0 to 1.0


async def generate_support_faq(topic: str):
    """Generate a customer support FAQ document with progress updates."""

    yield ProgressUpdate(
        stage="analyzing",
        message=f"Analyzing common questions about '{topic}'...",
        progress=0.15,
    )
    await asyncio.sleep(0.1)

    yield ProgressUpdate(
        stage="researching",
        message="Researching policy details and edge cases...",
        progress=0.40,
    )
    await asyncio.sleep(0.1)

    yield ProgressUpdate(
        stage="drafting",
        message="Generating FAQ content...",
        progress=0.65,
    )

    # Real LLM call
    prompt = f"""Create a concise customer support FAQ for the topic: "{topic}"

Format as exactly 3 questions and answers. Each answer should be 1-2 sentences.
Be specific and practical."""

    result = await llm.ainvoke([HumanMessage(content=prompt)])
    faq_content = result.content

    yield ProgressUpdate(
        stage="finalizing",
        message="Formatting and finalizing document...",
        progress=0.90,
    )
    await asyncio.sleep(0.05)

    yield ProgressUpdate(stage="complete", message="FAQ document ready!", progress=1.0)

    # Final result payload
    yield {"type": "result", "content": faq_content}


print("generate_support_faq() defined — async generator with progress updates")


In [ ]:
def progress_bar(progress: float, width: int = 28) -> str:
    filled = int(width * progress)
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {progress * 100:.0f}%"


async def run_progress_demo():
    topic = "product returns and refunds"
    print("=" * 60)
    print(f"GENERATING SUPPORT FAQ: '{topic}'")
    print("=" * 60)

    faq_result = None

    async for update in generate_support_faq(topic):
        if isinstance(update, ProgressUpdate):
            print(f"\n{progress_bar(update.progress)}")
            print(f"  Stage : {update.stage}")
            print(f"  Status: {update.message}")
        elif isinstance(update, dict) and update.get("type") == "result":
            faq_result = update["content"]

    print("\n" + "=" * 60)
    print("GENERATED FAQ DOCUMENT")
    print("=" * 60)
    print(faq_result)


asyncio.get_event_loop().run_until_complete(run_progress_demo())


## Frontend UX Patterns

Great UX for AI applications requires careful attention to loading states, accessibility, and feedback mechanisms. Key considerations:

- **Loading states** — progress indicators with stage tracking keep users informed during generation
- **Accessible inputs** — ARIA labels, keyboard navigation, and proper `htmlFor`/`id` pairings support all users
- **Chat message rendering** — visually distinguish user and AI messages; render markdown in AI responses
- **Feedback mechanisms** — thumbs up/down on responses provide training signal for system improvement

In [ ]:
STAGES = [
    {"key": "analyzing",  "label": "Analyzing your request"},
    {"key": "retrieving", "label": "Finding relevant information"},
    {"key": "generating", "label": "Generating response"},
]


def get_stage_class(stage_key: str, current_stage: str, stages: list[dict]) -> str:
    """Determine display class for a loading stage (Python translation of JS logic)."""
    stage_keys = [s["key"] for s in stages]
    try:
        current_index = stage_keys.index(current_stage)
        item_index = stage_keys.index(stage_key)
        if item_index < current_index:
            return "stage-complete"
        elif item_index == current_index:
            return "stage-active"
        else:
            return "stage-pending"
    except ValueError:
        return "stage-pending"


print("Stage progression through each active stage:")
for active_stage in ["analyzing", "retrieving", "generating"]:
    print(f"\n  Active: '{active_stage}'")
    for stage in STAGES:
        css_class = get_stage_class(stage["key"], active_stage, STAGES)
        icon = {"stage-complete": "✓", "stage-active": "▶", "stage-pending": "○"}[css_class]
        print(f"    {icon} [{css_class:<16}] {stage['label']}")

print()
print("=" * 55)
print("FEEDBACK COLLECTION DEMO")
print("=" * 55)


@dataclass
class MessageFeedback:
    message_id: str
    rating: str          # positive, negative
    feedback_text: str = ""
    created_at: datetime = field(default_factory=datetime.utcnow)


class FeedbackCollector:
    """Collects and aggregates user feedback on AI responses."""

    def __init__(self):
        self.feedback: list[MessageFeedback] = []

    def record(self, feedback: MessageFeedback):
        self.feedback.append(feedback)

    def summary(self) -> dict:
        total = len(self.feedback)
        if total == 0:
            return {"total": 0, "positive_rate": 0.0}
        positive = sum(1 for f in self.feedback if f.rating == "positive")
        return {
            "total": total,
            "positive": positive,
            "negative": total - positive,
            "positive_rate": f"{positive / total * 100:.1f}%",
        }


collector = FeedbackCollector()
collector.record(MessageFeedback("msg-001", "positive", "Very helpful!"))
collector.record(MessageFeedback("msg-002", "negative", "Didn't answer my question"))
collector.record(MessageFeedback("msg-003", "positive"))
collector.record(MessageFeedback("msg-004", "positive", "Clear and concise"))

print(f"Feedback summary: {collector.summary()}")
for fb in collector.feedback:
    icon = "👍" if fb.rating == "positive" else "👎"
    print(f"  {icon} {fb.message_id}: {fb.feedback_text or '(no comment)'}")


## Database Schema Design for Agent Apps

AI applications need schemas designed for multi-tenancy from day one. The key principle: **every piece of user-specific data should have a `user_id` foreign key**. Adding authentication later becomes a configuration change, not a rewrite.

Key columns to include beyond the basics:
- `tokens_used` and `model` on messages — for cost tracking and model versioning
- `feedback` and `feedback_text` on messages — for quality improvement loops
- `input_hash` on generated content — cache lookup to avoid regenerating identical requests
- `status` and `error_message` on documents — for async indexing workflows

In [ ]:
def generate_uuid():
    return str(uuid.uuid4())


Base = declarative_base()


class User(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=generate_uuid)
    email = Column(String, unique=True, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    conversations = relationship("Conversation", back_populates="user")
    generated_content = relationship("GeneratedContent", back_populates="user")


class Conversation(Base):
    __tablename__ = "conversations"
    id = Column(String, primary_key=True, default=generate_uuid)
    user_id = Column(String, ForeignKey("users.id"), nullable=False)
    title = Column(String)
    started_at = Column(DateTime, default=datetime.utcnow)
    last_activity = Column(DateTime, default=datetime.utcnow)
    message_count = Column(Integer, default=0)
    user = relationship("User", back_populates="conversations")
    messages = relationship(
        "Message", back_populates="conversation", order_by="Message.created_at"
    )


class Message(Base):
    __tablename__ = "messages"
    id = Column(String, primary_key=True, default=generate_uuid)
    conversation_id = Column(String, ForeignKey("conversations.id"), nullable=False)
    role = Column(String, nullable=False)
    content = Column(Text, nullable=False)
    tokens_used = Column(Integer)
    model = Column(String)
    feedback = Column(String)
    feedback_text = Column(Text)
    created_at = Column(DateTime, default=datetime.utcnow)
    conversation = relationship("Conversation", back_populates="messages")


class GeneratedContent(Base):
    __tablename__ = "generated_content"
    id = Column(String, primary_key=True, default=generate_uuid)
    user_id = Column(String, ForeignKey("users.id"), nullable=False)
    content_type = Column(String, nullable=False)
    title = Column(String)
    content = Column(Text, nullable=False)
    input_hash = Column(String, index=True)
    model = Column(String)
    tokens_used = Column(Integer)
    created_at = Column(DateTime, default=datetime.utcnow)
    user = relationship("User", back_populates="generated_content")


# Create all tables in SQLite in-memory database
engine = create_engine("sqlite:///:memory:", echo=False)
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)

print("Database tables created:")
for table_name in Base.metadata.tables:
    print(f"  ✓ {table_name}")


In [ ]:
session = SessionLocal()

# Create user
user = User(id=generate_uuid(), email="jane@example.com")
session.add(user)
session.flush()

# Create conversation
convo = Conversation(
    id=generate_uuid(),
    user_id=user.id,
    title="Return Request — Order #12345",
)
session.add(convo)
session.flush()

# Simulate a customer support exchange
exchanges = [
    ("user",      "I want to return order #12345. The product arrived damaged."),
    ("assistant", "I'm sorry to hear that! I've logged a return for order #12345. "
                  "You'll receive a prepaid shipping label within 24 hours. "
                  "Would you prefer a replacement or a refund?"),
    ("user",      "A full refund please."),
    ("assistant", "Done — a refund of $49.99 has been initiated and will appear "
                  "on your statement within 3-5 business days. Anything else I can help with?"),
]

for role, content in exchanges:
    msg = Message(
        id=generate_uuid(),
        conversation_id=convo.id,
        role=role,
        content=content,
        model="gpt-4o-mini" if role == "assistant" else None,
        tokens_used=len(content.split()) * 2 if role == "assistant" else None,
    )
    session.add(msg)

convo.message_count = len(exchanges)
convo.last_activity = datetime.utcnow()

# Store generated content with input_hash for caching
faq_input = "customer support FAQ for product returns and refunds"
input_hash = hashlib.sha256(faq_input.encode()).hexdigest()
generated = GeneratedContent(
    id=generate_uuid(),
    user_id=user.id,
    content_type="faq",
    title="Returns & Refunds FAQ",
    content="Q: How do I start a return?\nA: Contact support with your order number...",
    input_hash=input_hash,
    model="gpt-4o-mini",
)
session.add(generated)
session.commit()

# ── Query and display conversation history ─────────────────────────────────
print("=" * 60)
saved_convo = session.get(Conversation, convo.id)
print(f"CONVERSATION: {saved_convo.title}")
print(f"Messages: {saved_convo.message_count} | Last activity: {saved_convo.last_activity:%Y-%m-%d %H:%M}")
print("=" * 60)

for msg in saved_convo.messages:
    role_label = "Customer      " if msg.role == "user" else "Support Agent"
    print(f"\n[{role_label}]")
    print(f"  {msg.content}")
    if msg.tokens_used:
        print(f"  ({msg.tokens_used} tokens | model: {msg.model})")

# ── Cache lookup by input_hash ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("CACHE LOOKUP DEMO")
print("=" * 60)

lookup_hash = hashlib.sha256(faq_input.encode()).hexdigest()
cached = session.query(GeneratedContent).filter_by(input_hash=lookup_hash).first()

if cached:
    print(f"Cache HIT for: '{faq_input[:45]}...'")
    print(f"  Type  : {cached.content_type}")
    print(f"  Title : {cached.title}")
    print(f"  Saved : {cached.created_at:%Y-%m-%d %H:%M}")
    print(f"  Model : {cached.model}")
else:
    print("Cache MISS — would call LLM and generate new content")

session.close()


## Summary

This notebook demonstrated the core patterns for building full-stack AI applications:

- **Service layer** — wrap agents in an `AgentService` interface that isolates backend routes from LLM implementation details
- **SSE streaming** — use `StreamingResponse` with async generators and JSON-encoded chunks to preserve formatting over HTTP
- **WebSocket management** — `ConnectionManager` tracks active connections and routes messages by user ID
- **Background jobs** — accept requests immediately, return a job ID, and let clients poll for completion
- **Optimistic state updates** — update the UI instantly, roll back gracefully on failure
- **Retry logic** — `tenacity` with exponential backoff handles transient failures automatically
- **Progress updates** — async generators that yield `ProgressUpdate` milestones keep users engaged during long operations
- **Frontend UX** — loading stages, accessible inputs, and feedback collection improve perceived quality
- **Database schema** — isolate data by `user_id` from day one; use `input_hash` for content caching

These patterns are domain-agnostic — the same service layer, streaming infrastructure, background job system, and database schema apply to customer support, content generation, knowledge management, or any other AI application.